# ECVs: from a variable name to one combined dataset, through NERC and Beacon

Built from the [QCDIS/ECVs](https://github.com/QCDIS/ECVs) virtual lab. You give
the name of one of the 55 GCOS Essential Climate Variables, for example
`surface temperature`, `ozone` or `precipitation`, or its NERC code (`EXV002`),
plus a region, a period and a height range. The workflow then:

1. takes the variable name and the window (**input**);
2. finds the Essential Variable in the NERC Vocabulary Server collection
   [EXV](https://vocab.nerc.ac.uk/collection/EXV/current/) and its CF standard
   names through SPARQL: from its I-ADOPT components via the P01 and P02
   parameters, or, for the ECVs NVS has not decomposed yet, from its label
   (**NERC search**);
3. looks through the research-infrastructure tables served by the MARIS Beacon
   nodes for columns whose `standard_name` is one of those CF names, and checks
   that each has data in the window (**RI discovery and validation**);
4. downloads every selected source (**download**);
5. converts units, flags fill and implausible values, and plots the datasets per
   CF name (**validation**);
6. combines all valid rows into one CSV with its provenance (**combine**).

When no token-free research infrastructure serves the ECV (most ocean and land
ECVs: Argo and SeaDataNet CDI need a MARIS token), the run still completes, with
empty tables and a provenance file that says so.

The first cell holds the configuration (`conf_` values, including the registry
of RI tables) and the defaults of the workflow parameters. It is not
containerised: NaaVRE copies the `conf_` values into each cell, and the
`param_` values become workflow parameters of the input cell.


In [ ]:
# DO NOT CONTAINERISE, v2
# =====
# Configuration of the ECVs workflow

import os

conf_vlab_name = "ECVs"

# Shared storage, mounted in JupyterLab and in every workflow pod.
# Results go to <root>/<EXV code>/, e.g. naa-vre-user-data/ECVs/EXV002/.
conf_dir_workspace = os.path.join("/", "home", "jovyan", "Cloud Storage")
conf_minio_user_local_root = os.path.join(conf_dir_workspace, "naa-vre-user-data", conf_vlab_name)

# NERC Vocabulary Server
conf_SERVICE_URL_NERC_SPARQL = "https://vocab.nerc.ac.uk/sparql/sparql"
conf_SERVICE_URL_NERC_VOCAB = "http://vocab.nerc.ac.uk/collection"

# Words of an EXV label that qualify rather than name the variable; the label
# fallback (EXVs without I-ADOPT components) leaves them out. British spellings
# are mapped to the CF (US) spelling.
conf_label_qualifiers = ["surface", "upper", "subsurface", "and", "other", "the", "of", "from",
                         "properties", "budget", "earth", "ocean", "anthropogenic", "ad"]
# EXVs whose label words never occur in a CF name (NVS has no I-ADOPT
# decomposition for them either): the CF words to search for instead. Each
# inner list is one alternative; a CF name must contain all its words.
conf_exv_label_hints = {
    "EXV008": [["wind", "speed"], ["wind", "direction"], ["eastward", "wind"], ["northward", "wind"]],
    "EXV031": [["cfc11"], ["cfc12"], ["cfc113"], ["sulfur", "hexafluoride"]],
    "EXV034": [["phytoplankton"], ["zooplankton"]],
    "EXV038": [["river"], ["rivers"]],
    "EXV042": [["land", "ice"]],
    "EXV043": [["ice", "sheet"], ["ice", "shelf"]],
    "EXV045": [["photosynthetic", "absorbed"]],
    "EXV049": [["vegetation", "carbon", "content"]],
    "EXV053": [["anthropogenic", "emission"]],
    "EXV055": [["irrigation"]],
}
# I-ADOPT objects of interest that are a medium, not a substance; the others
# (methane, ozone, wind, oxygen, ...) restrict the CF names to those naming them.
conf_iadopt_media = ["water body", "air", "atmosphere", "sea water", "water"]
# I-ADOPT matrix -> words a CF name must contain (keeps ocean ECVs off "..._in_air")
conf_iadopt_matrix_words = {"water body": ["water"]}
# Platform and instrument quantities that carry ambient-sounding CF names
conf_cf_exclude = ["stagnation_air_temperature", "platform_speed_wrt_air", "platform_speed_wrt_ground",
                   "barometric_altitude", "gps_altitude", "altitude_wrt_radar", "status_flag"]
conf_label_spelling = {"vapour": "vapor", "colour": "color", "sulphate": "sulfate",
                       "sulphur": "sulfur", "aerosols": "aerosol"}

# Research-infrastructure tables on the MARIS Beacon nodes, in order of
# preference within each RI. NERC tells which variable to look for; this
# registry tells where each RI keeps time, position, height and site. The RI
# discovery step finds the variable's columns itself, from their standard_name.
# height_is_text: the height is a text attribute ("125.0 m"), filtered after
# download because Beacon would compare it as a string.
# matrix: the per-variable attribute naming the medium measured, if any.
# screen_all: test every column of the table (small tables with abbreviated
# names such as IAGOS O3_P1); otherwise only columns whose name words all occur
# in one of the CF names are tested.
conf_ri_registry = [
    {"ri": "ACTRIS", "node": "https://beacon-iriscc.maris.nl", "table": "actris-in-situ",
     "time": "time", "latitude": ".geospatial_lat_min", "longitude": ".geospatial_lon_min",
     "height": ".ebas_station_altitude", "height_is_text": True, "site": ".ebas_station_code",
     "matrix": "{column}.ebas_matrix", "screen_all": False, "token_required": False},
    {"ri": "ACTRIS", "node": "https://beacon-iriscc.maris.nl", "table": "actris-nrt",
     "time": "time", "latitude": ".geospatial_lat_min", "longitude": ".geospatial_lon_min",
     "height": ".ebas_station_altitude", "height_is_text": True, "site": ".ebas_station_code",
     "matrix": "{column}.ebas_matrix", "screen_all": True, "token_required": False},
    {"ri": "IAGOS", "node": "https://beacon-iriscc.maris.nl", "table": "iagos-l2",
     "time": "UTC_time", "latitude": "lat", "longitude": "lon",
     "height": "baro_alt_AC", "height_is_text": False, "site": "flight_name",
     "matrix": None, "screen_all": True, "token_required": False},
    {"ri": "IAGOS", "node": "https://beacon-iriscc.maris.nl", "table": "iagos-l1",
     "time": "UTC_time", "latitude": "lat", "longitude": "lon",
     "height": "baro_alt_AC", "height_is_text": False, "site": "flight_name",
     "matrix": None, "screen_all": True, "token_required": False},
    {"ri": "Argo", "node": "https://beacon-argo.maris.nl", "table": "argo",
     "time": "JULD", "latitude": "LATITUDE", "longitude": "LONGITUDE",
     "height": "PRES", "height_is_text": False, "site": "PLATFORM_NUMBER",
     "matrix": None, "screen_all": False, "token_required": True},
    {"ri": "CDI", "node": "https://beacon-cdi.maris.nl", "table": "default",
     "time": "TIME", "latitude": "LATITUDE", "longitude": "LONGITUDE",
     "height": "DEPTH", "height_is_text": False, "site": "SDN_STATION",
     "matrix": None, "screen_all": False, "token_required": True},
]

# Which RI matrix values hold ambient data. Every instrument logs its own
# temperature, pressure and humidity, which ACTRIS tags with the same CF names
# as the weather station's; for those state variables only the weather station
# ("met") is ambient. For everything else, only "instrument" is left out.
conf_ri_matrix_rules = {
    "exclude": ["instrument"],
    "only": {
        "air_temperature": ["met"], "air_pressure": ["met"], "relative_humidity": ["met"],
        "wind_speed": ["met"], "wind_from_direction": ["met"], "wind_speed_of_gust": ["met"],
        "lwe_thickness_of_precipitation_amount": ["met"],
    },
}

# At most this many sources are downloaded (CF-name tier first, then RI order)
conf_max_sources = 12

conf_delimiter_csv = ","

# Sentinels that mark missing values in the RIs' files; a row's own
# _FillValue / missing_value attribute is checked as well.
conf_fill_values = [9999.99, 9999.9, 99999.0, -9999.0, -999.0, -999.9, -9726.75, 9.96921e36]
# Plausible range per CF standard name: [canonical unit, min, max]
conf_valid_ranges = {
    "air_temperature": ["degC", -90.0, 60.0],
    "surface_temperature": ["degC", -90.0, 70.0],
    "sea_water_temperature": ["degC", -3.0, 40.0],
    "sea_surface_temperature": ["degC", -3.0, 40.0],
    "air_pressure": ["hPa", 100.0, 1100.0],
    "surface_air_pressure": ["hPa", 400.0, 1100.0],
    "relative_humidity": ["%", 0.0, 105.0],
    "wind_speed": ["m/s", 0.0, 80.0],
    "wind_speed_of_gust": ["m/s", 0.0, 100.0],
    "wind_from_direction": ["deg", 0.0, 360.0],
    "eastward_wind": ["m/s", -150.0, 150.0],
    "northward_wind": ["m/s", -150.0, 150.0],
    "lwe_thickness_of_precipitation_amount": ["mm", 0.0, 1000.0],
    "mole_fraction_of_ozone_in_air": ["ppb", 0.0, 15000.0],
    "mole_fraction_of_carbon_monoxide_in_air": ["ppb", 0.0, 10000.0],
    "mole_fraction_of_carbon_dioxide_in_air": ["ppm", 300.0, 700.0],
    "mole_fraction_of_methane_in_air": ["ppb", 1000.0, 5000.0],
    "mole_fraction_of_nitrogen_dioxide_in_air": ["ppb", 0.0, 1000.0],
    "mole_fraction_of_water_vapor_in_air": ["ppm", 0.0, 50000.0],
}

# Workflow parameters (defaults)
param_variable_name = "surface temperature"
param_lon_min = 0.0
param_lon_max = 20.0
param_lat_min = 45.0
param_lat_max = 55.0
param_start_date = "2024-01-01"
param_end_date = "2024-12-31"
param_height_min = 0.0
param_height_max = 200.0


## 1. Input: the variable and the window


In [ ]:
# ECVs variable input v2
# ---
# NaaVRE:
#  cell:
#   params:
#    - param_variable_name:
#        type: String
#        default_value: "surface temperature"
#    - param_lon_min:
#        type: Float
#        default_value: 0.0
#    - param_lon_max:
#        type: Float
#        default_value: 20.0
#    - param_lat_min:
#        type: Float
#        default_value: 45.0
#    - param_lat_max:
#        type: Float
#        default_value: 55.0
#    - param_start_date:
#        type: String
#        default_value: "2024-01-01"
#    - param_end_date:
#        type: String
#        default_value: "2024-12-31"
#    - param_height_min:
#        type: Float
#        default_value: 0.0
#    - param_height_max:
#        type: Float
#        default_value: 200.0
#   outputs:
#    - variable_name: String
#    - lon_min: Float
#    - lon_max: Float
#    - lat_min: Float
#    - lat_max: Float
#    - start_date: String
#    - end_date: String
#    - height_min: Float
#    - height_max: Float
# ...
import datetime

variable_name = " ".join(str(param_variable_name).split())
if not variable_name:
    raise ValueError("param_variable_name is empty: give a variable name or an EXV code")

lon_min, lon_max = float(param_lon_min), float(param_lon_max)
lat_min, lat_max = float(param_lat_min), float(param_lat_max)
height_min, height_max = float(param_height_min), float(param_height_max)
start_date, end_date = str(param_start_date).strip(), str(param_end_date).strip()

if not (-180.0 <= lon_min < lon_max <= 180.0):
    raise ValueError(f"Longitude range {lon_min}..{lon_max} is not valid")
if not (-90.0 <= lat_min < lat_max <= 90.0):
    raise ValueError(f"Latitude range {lat_min}..{lat_max} is not valid")
if not height_min <= height_max:
    raise ValueError(f"Height range {height_min}..{height_max} is not valid")
if datetime.date.fromisoformat(start_date) > datetime.date.fromisoformat(end_date):
    raise ValueError(f"Period {start_date}..{end_date} is not valid")

print(f"Variable: {variable_name!r}")
print(f"Window: lon {lon_min}..{lon_max}, lat {lat_min}..{lat_max}, "
      f"{start_date}..{end_date}, height {height_min}..{height_max} m")


## 2. NERC Vocabulary Server: the Essential Variable and its CF standard names


In [ ]:
# ECVs NERC EXV search v2
# ---
# NaaVRE:
#  cell:
#   inputs:
#    - variable_name: String
#   outputs:
#    - exv_code: String
#    - file_exv: String
# ...
import json
import os
import re
import time

try:
    import requests
except ImportError as exc:
    raise RuntimeError("requests must be installed") from exc

PREFIXES = '''
PREFIX skos: <http://www.w3.org/2004/02/skos/core#>
PREFIX iadopt: <https://w3id.org/iadopt/ont#>
PREFIX owl: <http://www.w3.org/2002/07/owl#>
'''
NVS = conf_SERVICE_URL_NERC_VOCAB


def sparql(query):
    # Run a SPARQL query on the NVS endpoint; the bindings as plain dicts
    for attempt in range(4):
        try:
            response = requests.get(
                conf_SERVICE_URL_NERC_SPARQL,
                params={"query": PREFIXES + query},
                headers={"Accept": "application/sparql-results+json"},
                timeout=180,
            )
            response.raise_for_status()
            return [{k: v["value"] for k, v in row.items()}
                    for row in response.json()["results"]["bindings"]]
        except requests.RequestException:
            if attempt == 3:
                raise
            time.sleep(5 * (attempt + 1))


def code_of(uri):
    return uri.rstrip("/").split("/")[-1]


def words(text):
    return re.sub(r"[^a-z0-9]+", " ", text.lower()).split()


def core_words(label):
    # The words that name the variable: no qualifiers, CF (US) spelling, singular
    out = []
    for w in words(label):
        w = conf_label_spelling.get(w, w)
        if w in conf_label_qualifiers or len(w) < 3:
            continue
        if len(w) > 5 and w.endswith(("xes", "ches", "shes", "sses")):
            w = w[:-2]
        elif len(w) > 4 and w.endswith("s") and not w.endswith(("ss", "us", "is")):
            w = w[:-1]
        out.append(w)
    return out


# 1. All Essential Variables (EXV collection), then pick the best match
rows = sparql(f'''
SELECT ?exv ?label ?alt ?definition ?deprecated WHERE {{
  <{NVS}/EXV/current/> skos:member ?exv .
  ?exv skos:prefLabel ?label .
  OPTIONAL {{ ?exv skos:altLabel ?alt }}
  OPTIONAL {{ ?exv skos:definition ?definition }}
  OPTIONAL {{ ?exv owl:deprecated ?deprecated }}
}}''')
exvs = {}
for row in rows:
    entry = exvs.setdefault(row["exv"], {
        "code": code_of(row["exv"]), "uri": row["exv"], "label": row["label"],
        "alt_labels": set(), "definition": row.get("definition", ""),
        "deprecated": row.get("deprecated", "false") == "true"})
    if row.get("alt"):
        entry["alt_labels"].add(row["alt"])

query_words = words(variable_name)


def score(entry):
    if variable_name.upper() == entry["code"]:
        return 100
    label_words = words(entry["label"])
    if query_words == label_words:
        return 90
    if any(query_words == words(alt) for alt in entry["alt_labels"]):
        return 80
    if set(query_words) <= set(label_words):
        return 70 - (len(label_words) - len(query_words))
    if " ".join(query_words) in " ".join(label_words):
        return 50
    if set(core_words(variable_name)) and set(core_words(variable_name)) <= set(core_words(entry["label"])):
        return 40 - len(core_words(entry["label"]))
    return 0


ranked = sorted((e for e in exvs.values() if not e["deprecated"] and score(e) > 0),
                key=lambda e: (-score(e), e["code"]))
if not ranked:
    raise ValueError(f"No Essential Variable in NERC EXV matches {variable_name!r}")
exv = ranked[0]
exv_code = exv["code"]
exv_uri = exv["uri"]
print(f"{variable_name!r} -> {exv_code} {exv['label']!r}"
      f" (other matches: {[e['code'] + ' ' + e['label'] for e in ranked[1:6]]})")

# 2. Supporting details: related links and I-ADOPT components
details = sparql(f'''
SELECT ?p ?o ?label WHERE {{
  <{exv_uri}> ?p ?o .
  FILTER(STRSTARTS(STR(?p), "https://w3id.org/iadopt/ont#") || ?p = skos:related)
  OPTIONAL {{ ?o skos:prefLabel ?label . FILTER(LANG(?label) = "en" || LANG(?label) = "") }}
}}''')
iadopt = {"property": {}, "matrix": {}, "object_of_interest": {}}
related = set()
for row in details:
    p = row["p"]
    if p.endswith("hasApplicableProperty"):
        iadopt["property"].setdefault(row["o"], row.get("label", ""))
    elif p.endswith("hasApplicableMatrix"):
        iadopt["matrix"].setdefault(row["o"], row.get("label", ""))
    elif p.endswith("hasApplicableObjectOfInterest"):
        iadopt["object_of_interest"].setdefault(row["o"], row.get("label", ""))
    elif p.endswith("related"):
        related.add(row["o"])

# 3. P01 parameters with the same I-ADOPT property and the same object of
#    interest, or, when none share it, the same matrix (the exv_to_r03 pattern
#    of the ECV17 notebook, relaxed: NVS often records the P01 property only as
#    skos:broader). The object of interest comes first because "concentration
#    in the atmosphere" alone would match every gas and aerosol parameter.
def p01_pattern_for(ctx_uris):
    values = " ".join(f"<{u}>" for u in sorted(ctx_uris))
    return f'''
  <{exv_uri}> iadopt:hasApplicableProperty ?property .
  VALUES ?ctx {{ {values} }}
  <{NVS}/P01/current/> skos:member ?p01 .
  ?p01 iadopt:hasProperty|skos:broader ?property .
  ?p01 iadopt:hasMatrix|iadopt:hasObjectOfInterest ?ctx .
'''


p01 = {}
mapped = []
p01_pattern = ""
for ctx_uris in (iadopt["object_of_interest"], iadopt["matrix"]):
    if not iadopt["property"] or not ctx_uris:
        continue
    p01_pattern = p01_pattern_for(ctx_uris)
    p01 = {r["p01"]: r["label"] for r in sparql(f'''
SELECT DISTINCT ?p01 ?label WHERE {{ {p01_pattern} ?p01 skos:prefLabel ?label . }}''')}
    if p01:
        break
if p01:
    # 4. Terms mapped from those P01 parameters: P02 discovery parameters,
    #    CF standard names (P07) and Argo parameter codes (R03)
    mapped = sparql(f'''
SELECT DISTINCT ?collection ?term ?label WHERE {{
  {p01_pattern}
  ?p01 owl:sameAs|skos:related|skos:broader ?term .
  ?collection skos:member ?term .
  FILTER(?collection IN (<{NVS}/P02/current/>, <{NVS}/P07/current/>, <{NVS}/R03/current/>))
  ?term skos:prefLabel ?label .
}}''')
terms = {"P02": {}, "P07": {}, "R03": {}}
for row in mapped:
    terms[code_of(row["collection"].rstrip("/").rsplit("/", 1)[0])][row["term"]] = row["label"]
cf_tier = {label.strip(): 1 for label in terms["P07"].values()}

# 5. CF standard names under the P02 discovery parameters (NVS P02 -> P07 mapping)
if terms["P02"]:
    values = " ".join(f"<{u}>" for u in sorted(terms["P02"]))
    for row in sparql(f'''
SELECT DISTINCT ?term ?label WHERE {{
  VALUES ?p02 {{ {values} }}
  ?p02 skos:narrower|skos:related ?term .
  <{NVS}/P07/current/> skos:member ?term .
  ?term skos:prefLabel ?label .
}}'''):
        cf_tier.setdefault(row["label"].strip(), 2)

# 6. EXVs that NVS has not decomposed (no CF name yet): the CF names whose
#    words include every core word of the EXV label, e.g. "Surface water
#    vapour" -> water, vapor; or, for labels that never occur in CF, the words
#    of conf_exv_label_hints. Each I-ADOPT object of interest that names a
#    substance or phenomenon (e.g. "methane", "wind") also adds the CF names
#    that contain all its words, so a multi-component EXV keeps every component.
label_core = core_words(exv["label"])
media = set(conf_iadopt_media)
substance_cores = [core_words(lab) for lab in iadopt["object_of_interest"].values()
                   if lab and lab.lower() not in media and core_words(lab)]
hints = conf_exv_label_hints.get(exv_code, [])
fallback_used = not cf_tier and bool(label_core or hints)
word_sets = (([label_core] if label_core else []) + hints if fallback_used else []) + substance_cores
all_cf = []
if word_sets or cf_tier:
    all_cf = sorted({row["label"].strip() for row in sparql(f'''
SELECT ?label WHERE {{ <{NVS}/P07/current/> skos:member ?term . ?term skos:prefLabel ?label . }}''')})


def cf_words(name):
    return set(name.lower().replace("-", "_").split("_"))


for name in all_cf:
    if any(all(w in cf_words(name) for w in ws) for ws in word_sets):
        cf_tier.setdefault(name, 3)


# 7. Constraints, each kept only when it leaves at least one CF name: the
#    substance objects of interest, the matrix words, and no platform quantities
def constrain(names, keep):
    kept = {n: t for n, t in names.items() if keep(n)}
    return kept if kept else names


if substance_cores:
    cf_tier = constrain(cf_tier, lambda n: any(all(w in cf_words(n) for w in ws) for ws in substance_cores))
matrix_words = [w for m in iadopt["matrix"].values() for w in conf_iadopt_matrix_words.get(m.lower(), [])]
if matrix_words:
    cf_tier = constrain(cf_tier, lambda n: any(w in cf_words(n) for w in matrix_words))
cf_tier = {n: t for n, t in cf_tier.items() if n not in conf_cf_exclude}

cf_names = sorted(cf_tier)
result = {
    "query": variable_name,
    "exv": {"code": exv_code, "uri": exv_uri, "label": exv["label"],
            "alt_labels": sorted(exv["alt_labels"]), "definition": exv["definition"],
            "related": sorted(related),
            "iadopt": {k: [{"uri": u, "code": code_of(u), "label": lab} for u, lab in sorted(v.items())]
                       for k, v in iadopt.items()}},
    "other_matches": [{"code": e["code"], "label": e["label"]} for e in ranked[1:]],
    "p01_parameters": [{"uri": u, "code": code_of(u), "label": lab} for u, lab in sorted(p01.items())],
    "p02_discovery_parameters": [{"uri": u, "code": code_of(u), "label": lab}
                                 for u, lab in sorted(terms["P02"].items())],
    "cf_standard_names": cf_names,
    "cf_standard_name_tiers": {name: cf_tier[name] for name in cf_names},
    "cf_source": (("label hint (conf_exv_label_hints)" if hints else "label fallback (no I-ADOPT in NVS)")
                  if fallback_used and cf_names
                  else "I-ADOPT -> P01 -> P02/P07" if cf_names else "none"),
    "label_core_words": label_core,
    "label_hints": hints,
    "substance_objects_of_interest": substance_cores,
    "argo_r03_codes": sorted({code_of(u) for u in terms["R03"]}),
}

out_dir = os.path.join(conf_minio_user_local_root, exv_code)
os.makedirs(out_dir, exist_ok=True)
file_exv = os.path.join(out_dir, f"{conf_vlab_name}-1-NERC.{exv_code}.json")
with open(file_exv, "w", encoding="utf-8", newline="\n") as fh:
    json.dump(result, fh, indent=2, ensure_ascii=False)
    fh.write("\n")

print(f"I-ADOPT: { {k: list(v.values()) for k, v in iadopt.items()} }")
print(f"P01 parameters: {len(p01)}, P02: {sorted(terms['P02'].values())}")
print(f"CF standard names ({len(cf_names)}, {result['cf_source']}): "
      f"tier 1 {[n for n in cf_names if cf_tier[n] == 1][:20]}")
print(f"Argo R03 codes: {result['argo_r03_codes']}")
print(f"Output: {file_exv}")


## 3. Research infrastructures: find the variable's columns and check the data exist


In [ ]:
# ECVs RI discovery v2
# ---
# NaaVRE:
#  cell:
#   inputs:
#    - exv_code: String
#    - file_exv: String
#    - lon_min: Float
#    - lon_max: Float
#    - lat_min: Float
#    - lat_max: Float
#    - start_date: String
#    - end_date: String
#    - height_min: Float
#    - height_max: Float
#   outputs:
#    - file_sources: String
#    - file_discovery: String
# ...
import json
import os
import re
import time
import importlib
from concurrent.futures import ThreadPoolExecutor

try:
    import requests
    import pandas as pd
    beacon_api = importlib.import_module("beacon_api")
except ImportError as exc:
    raise RuntimeError("requests, pandas and beacon-api must be installed") from exc

with open(file_exv, encoding="utf-8") as fh:
    nerc = json.load(fh)
cf_tier = nerc["cf_standard_name_tiers"]
cf_names = set(cf_tier)
r03_codes = set(nerc["argo_r03_codes"])
cf_word_sets = [set(name.lower().replace("-", "_").split("_")) for name in cf_names]


def with_retry(call, tries=4):
    # Beacon nodes are shared services: retry transient errors with a backoff
    for attempt in range(tries):
        try:
            return call()
        except Exception:
            if attempt == tries - 1:
                raise
            time.sleep(5 * (attempt + 1))
NOT_DATA = ("_qc", "_flag", "_error", "_process_flag", "_validity_flag", "_status",
            "_stddev", "_precision", "_accuracy", "_uncertainty", "_det_lim", "_ExpUnc2s", "_ExpUnc3s")
UNIT_WORDS = {"ug", "ng", "pg", "mg", "per", "m3", "l", "mol", "nmol", "pmol", "umol", "n", "s",
              "amean", "median", "min", "max", "perc8413", "prec1587", "fraction"}


def column_words(name):
    return [w for w in re.split("[-_ ]+", name.lower()) if w and w not in UNIT_WORDS and not w.isdigit()]


def screened(name, entry):
    # Columns worth reading the standard_name of
    if name in cf_names or name in r03_codes:
        return True
    if entry["screen_all"]:
        return True
    cw = set(column_words(name))
    return bool(cw) and any(cw <= ws for ws in cf_word_sets)


def matrix_filter(query, entry, column, standard_name):
    # Add the ambient-matrix condition of conf_ri_matrix_rules; returns (query, description)
    if not entry.get("matrix"):
        return query, ""
    query = query.add_select_column(entry["matrix"].format(column=column), "matrix")
    only = conf_ri_matrix_rules["only"].get(standard_name)
    if only:
        return (query.add_filter(beacon_api.OrFilter([beacon_api.EqualsFilter("matrix", v) for v in only])),
                "only " + ",".join(only))
    exclude = conf_ri_matrix_rules["exclude"]
    for v in exclude:
        query = query.add_filter(beacon_api.NotEqualsFilter("matrix", v))
    return query, "not " + ",".join(exclude)


def window_query(table, entry, column, standard_name):
    # The ECVs-Data.ipynb query for one column and one CF name, within the window
    query = table.query()
    query, _ = matrix_filter(query, entry, column, standard_name)
    query = (query
             .add_select_column(entry["time"], "TIME")
             .add_select_column(entry["latitude"], "LATITUDE")
             .add_select_column(entry["longitude"], "LONGITUDE")
             .add_select_column(entry["height"], "HEIGHT")
             .add_select_column(entry["site"], "site")
             .add_select_column(column, "value_raw")
             .add_select_column(f"{column}.units", "units_raw")
             .add_select_column(f"{column}.standard_name", "standard_name")
             .add_equals_filter("standard_name", standard_name)
             .add_range_filter("TIME", start_date, end_date)
             .add_range_filter("LONGITUDE", lon_min, lon_max)
             .add_range_filter("LATITUDE", lat_min, lat_max))
    if not entry["height_is_text"]:
        query = query.add_range_filter("HEIGHT", height_min, height_max)
    return query.add_is_not_null_filter("value_raw")


def blank(entry, status, **extra):
    row = {"ri": entry["ri"], "node": entry["node"], "table": entry["table"], "column": "",
           "standard_name": "", "tier": "", "units": "", "matrix": "", "rows_in_window": "", "status": status}
    row.update(extra)
    return row


report, found, clients = [], [], {}
for entry in conf_ri_registry:
    if entry["token_required"]:
        report.append(blank(entry, "skipped: the node needs a MARIS Beacon token"))
        continue
    if not cf_names:
        report.append(blank(entry, "skipped: NERC gives no CF standard name to look for"))
        continue
    def get_schema():
        response = requests.get(f"{entry['node']}/api/table-schema",
                                params={"table_name": entry["table"]}, timeout=300)
        response.raise_for_status()
        return {f["name"] for f in response.json()["fields"]}

    try:
        fields = with_retry(get_schema)
    except Exception as exc:
        report.append(blank(entry, f"error: table schema unreachable ({type(exc).__name__}, 4 tries)"))
        continue
    columns = sorted(
        (name for name in fields
         if "." not in name and f"{name}.standard_name" in fields and not name.endswith(NOT_DATA)
         and name not in (entry["time"], entry["latitude"], entry["longitude"], entry["height"], entry["site"])
         and screened(name, entry)),
        key=lambda name: (len(name), name))
    try:
        if entry["node"] not in clients:
            clients[entry["node"]] = with_retry(lambda: beacon_api.Client(entry["node"]))
        table = with_retry(lambda: clients[entry["node"]].list_tables())[entry["table"]]
    except Exception as exc:
        report.append(blank(entry, f"error: Beacon node unreachable ({type(exc).__name__}, 4 tries)"))
        continue

    def names_of(column):
        # 3a. the CF names and units a column carries
        try:
            df = with_retry(lambda: table.query()
                            .add_select_column(f"{column}.standard_name", "sn")
                            .add_select_column(f"{column}.units", "units")
                            .add_is_not_null_filter("sn")
                            .set_distinct(["sn", "units"])
                            .to_pandas_dataframe())
            return column, df.astype(str).values.tolist(), ""
        except Exception as exc:
            return column, [], str(exc)[:120]

    with ThreadPoolExecutor(max_workers=4) as pool:
        carried = list(pool.map(names_of, columns))

    # one source per (table, CF name): the first column (shortest name) with data
    by_name = {}
    for column, pairs, error in carried:
        if error:
            report.append(blank(entry, f"error reading standard_name: {error}", column=column))
            continue
        for sn in sorted({sn for sn, _ in pairs} & cf_names):
            units = ";".join(sorted({u for s, u in pairs if s == sn}))
            by_name.setdefault(sn, []).append((column, units))
    if not by_name:
        report.append(blank(entry, f"no column with a matching standard_name ({len(columns)} screened)"))
    for sn in sorted(by_name, key=lambda s: (cf_tier[s], s)):
        selected = None
        for column, units in by_name[sn]:
            _, matrix = matrix_filter(table.query(), entry, column, sn)
            row = blank(entry, "", column=column, standard_name=sn, tier=cf_tier[sn], units=units,
                        matrix=matrix)
            if selected:
                report.append(row | {"status": f"alternative to {selected}"})
                continue
            # 3b. does it hold data in the window?
            try:
                n = len(with_retry(lambda: window_query(table, entry, column, sn).set_limit(1).to_pandas_dataframe()))
            except Exception as exc:
                report.append(row | {"status": f"error in window query: {str(exc)[:120]}"})
                continue
            if not n:
                report.append(row | {"rows_in_window": 0, "status": "matches, but no data in the window"})
                continue
            selected = column
            report.append(row | {"rows_in_window": ">0", "status": "has data"})
            found.append(entry | {"column": column, "standard_name": sn, "tier": cf_tier[sn], "units": units,
                                  "matrix_rule": matrix,
                                  "fill_attrs": [a for a in ("_FillValue", "missing_value")
                                                 if f"{column}.{a}" in fields],
                                  "report_index": len(report) - 1})

# per RI and CF name, the first table in registry order; then at most conf_max_sources
order = {(e["node"], e["table"]): i for i, e in enumerate(conf_ri_registry)}
sources, covered = [], {}
for f in sorted(found, key=lambda f: (f["tier"], order[(f["node"], f["table"])], f["standard_name"])):
    key = (f["ri"], f["standard_name"])
    if key in covered:
        report[f["report_index"]]["status"] = f"has data; not downloaded, {f['ri']} {f['standard_name']} is covered by {covered[key]}"
        continue
    if len(sources) >= conf_max_sources:
        report[f["report_index"]]["status"] = f"has data; not downloaded, limit of {conf_max_sources} sources reached"
        continue
    covered[key] = f["table"]
    report[f["report_index"]]["status"] = "selected for download"
    sources.append({k: v for k, v in f.items() if k != "report_index"})

out_dir = os.path.join(conf_minio_user_local_root, exv_code)
os.makedirs(out_dir, exist_ok=True)
file_discovery = os.path.join(out_dir, f"{conf_vlab_name}-2-RI-discovery.{exv_code}.csv")
columns_out = ["ri", "node", "table", "column", "standard_name", "tier", "units", "matrix", "rows_in_window", "status"]
pd.DataFrame(report, columns=columns_out).to_csv(file_discovery, index=False)
file_sources = os.path.join(out_dir, f"{conf_vlab_name}-2-RI-sources.{exv_code}.json")
with open(file_sources, "w", encoding="utf-8", newline="\n") as fh:
    json.dump(sources, fh, indent=2)
    fh.write("\n")

shown = pd.DataFrame(report, columns=columns_out)
print(shown[shown["status"].str.startswith(("selected", "skipped", "no column"))]
      [["ri", "table", "column", "standard_name", "units", "matrix", "status"]].to_string(index=False))
if sources:
    print(f"Selected: {[(s['ri'], s['table'], s['column'], s['standard_name']) for s in sources]}")
else:
    print(f"No token-free research infrastructure has {exv_code} data in the window; "
          f"the next steps write empty results")


## 4. Download every selected source


In [ ]:
# ECVs data download v2
# ---
# NaaVRE:
#  cell:
#   inputs:
#    - exv_code: String
#    - file_sources: String
#    - lon_min: Float
#    - lon_max: Float
#    - lat_min: Float
#    - lat_max: Float
#    - start_date: String
#    - end_date: String
#    - height_min: Float
#    - height_max: Float
#   outputs:
#    - file_downloads: String
# ...
import json
import os
import re
import time
import importlib

try:
    import pandas as pd
    beacon_api = importlib.import_module("beacon_api")
except ImportError as exc:
    raise RuntimeError("pandas and beacon-api must be installed") from exc


def with_retry(call, tries=4):
    # Beacon nodes are shared services: retry transient errors with a backoff
    for attempt in range(tries):
        try:
            return call()
        except Exception:
            if attempt == tries - 1:
                raise
            time.sleep(5 * (attempt + 1))

with open(file_sources, encoding="utf-8") as fh:
    sources = json.load(fh)
out_dir = os.path.join(conf_minio_user_local_root, exv_code)
os.makedirs(out_dir, exist_ok=True)

downloads, clients = [], {}
for source in sources:
    if source["node"] not in clients:
        clients[source["node"]] = with_retry(lambda: beacon_api.Client(source["node"]))
    table = with_retry(lambda: clients[source["node"]].list_tables())[source["table"]]
    column, sn = source["column"], source["standard_name"]
    query = table.query()
    if source.get("matrix"):
        query = query.add_select_column(source["matrix"].format(column=column), "matrix")
        only = conf_ri_matrix_rules["only"].get(sn)
        if only:
            query = query.add_filter(beacon_api.OrFilter([beacon_api.EqualsFilter("matrix", v) for v in only]))
        else:
            for v in conf_ri_matrix_rules["exclude"]:
                query = query.add_filter(beacon_api.NotEqualsFilter("matrix", v))
    for attr in source["fill_attrs"]:
        query = query.add_select_column(f"{column}.{attr}", attr.strip("_").lower())
    query = (query
             .add_select_column(source["time"], "TIME")
             .add_select_column(source["latitude"], "LATITUDE")
             .add_select_column(source["longitude"], "LONGITUDE")
             .add_select_column(source["height"], "HEIGHT")
             .add_select_column(source["site"], "site")
             .add_select_column(column, "value_raw")
             .add_select_column(f"{column}.units", "units_raw")
             .add_select_column(f"{column}.standard_name", "standard_name")
             .add_equals_filter("standard_name", sn)
             .add_range_filter("TIME", start_date, end_date)
             .add_range_filter("LONGITUDE", lon_min, lon_max)
             .add_range_filter("LATITUDE", lat_min, lat_max))
    if not source["height_is_text"]:
        query = query.add_range_filter("HEIGHT", height_min, height_max)
    query = query.add_is_not_null_filter("value_raw")
    df = with_retry(query.to_pandas_dataframe)
    n_server = len(df)

    # A text height ("125.0 m") cannot be range-filtered on the server
    height = df["HEIGHT"]
    if source["height_is_text"]:
        height = pd.to_numeric(height.astype(str).str.replace("m", "", regex=False).str.strip(),
                               errors="coerce")
    df["HEIGHT"] = pd.to_numeric(height, errors="coerce")
    df = df[df["HEIGHT"].between(height_min, height_max)].copy()

    df.insert(0, "RI", source["ri"])
    df.insert(1, "table", source["table"])
    df.insert(2, "variable", column)
    df["site"] = df["site"].astype(str)
    # Beacon returns rows in no fixed order; sort so reruns are byte-identical
    df = df.sort_values(by=list(df.columns), kind="mergesort", na_position="last").reset_index(drop=True)

    safe = re.sub("[^A-Za-z0-9.-]+", "-", f"{column}.{sn}")
    path = os.path.join(out_dir, f"{conf_vlab_name}-4-{source['ri']}.{source['table']}.{safe}.{exv_code}.csv")
    df.to_csv(path, sep=conf_delimiter_csv, index=False)
    downloads.append({"ri": source["ri"], "table": source["table"], "column": column, "standard_name": sn,
                      "file": path, "rows_from_beacon": n_server, "rows_in_height_range": len(df)})
    print(f"{source['ri']} {source['table']}.{column} ({sn}): {n_server} rows from Beacon, "
          f"{len(df)} within {height_min}..{height_max} m")

file_downloads = os.path.join(out_dir, f"{conf_vlab_name}-4-downloads.{exv_code}.json")
with open(file_downloads, "w", encoding="utf-8", newline="\n") as fh:
    json.dump(downloads, fh, indent=2)
    fh.write("\n")
print(f"{len(downloads)} sources downloaded")


## 5. Validate: units, fill values, plausible range, plots


In [ ]:
# ECVs data validation v2
# ---
# NaaVRE:
#  cell:
#   inputs:
#    - exv_code: String
#    - file_exv: String
#    - file_downloads: String
#   outputs:
#    - file_validated: String
#    - file_validation: String
#    - file_map: String
#    - file_timeseries: String
#    - file_histogram: String
# ...
import json
import math
import os

try:
    import numpy as np
    import pandas as pd
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
except ImportError as exc:
    raise RuntimeError("numpy, pandas and matplotlib must be installed") from exc

with open(file_exv, encoding="utf-8") as fh:
    nerc = json.load(fh)
with open(file_downloads, encoding="utf-8") as fh:
    downloads = json.load(fh)
out_dir = os.path.join(conf_minio_user_local_root, exv_code)
os.makedirs(out_dir, exist_ok=True)
exv_label = nerc["exv"]["label"]

# unit as served (lower case) -> (canonical unit, scale, offset)
UNITS = {
    "k": ("degC", 1.0, -273.15), "kelvin": ("degC", 1.0, -273.15),
    "deg c": ("degC", 1.0, 0.0), "degc": ("degC", 1.0, 0.0), "degree_celsius": ("degC", 1.0, 0.0),
    "degrees_celsius": ("degC", 1.0, 0.0), "celsius": ("degC", 1.0, 0.0), "\u00b0c": ("degC", 1.0, 0.0),
    "hpa": ("hPa", 1.0, 0.0), "mbar": ("hPa", 1.0, 0.0), "pa": ("hPa", 0.01, 0.0),
    "ppb": ("ppb", 1.0, 0.0), "nmol/mol": ("ppb", 1.0, 0.0), "1e-9": ("ppb", 1.0, 0.0),
    "ppm": ("ppm", 1.0, 0.0), "umol/mol": ("ppm", 1.0, 0.0), "1e-6": ("ppm", 1.0, 0.0),
    "ppt": ("ppt", 1.0, 0.0), "pmol/mol": ("ppt", 1.0, 0.0),
    "m s-1": ("m/s", 1.0, 0.0), "m/s": ("m/s", 1.0, 0.0),
    "deg": ("deg", 1.0, 0.0), "degree": ("deg", 1.0, 0.0), "degrees": ("deg", 1.0, 0.0),
    "%": ("%", 1.0, 0.0), "percent": ("%", 1.0, 0.0),
    "mm": ("mm", 1.0, 0.0),
    "ug/m3": ("ug/m3", 1.0, 0.0), "\u00b5g/m3": ("ug/m3", 1.0, 0.0),
    "1/cm3": ("1/cm3", 1.0, 0.0), "cm-3": ("1/cm3", 1.0, 0.0), "no cm-3": ("1/cm3", 1.0, 0.0),
}


def canonical(values, units):
    # Values in the canonical unit of their served unit, and that unit, per row
    key = units.astype(str).str.strip().str.lower()
    unit = key.map(lambda k: UNITS.get(k, (None, 1.0, 0.0))[0])
    scale = key.map(lambda k: UNITS.get(k, (None, 1.0, 0.0))[1]).astype(float)
    offset = key.map(lambda k: UNITS.get(k, (None, 1.0, 0.0))[2]).astype(float)
    return (values.astype(float) * scale + offset).round(4), unit.fillna(units.astype(str).str.strip())


def is_fill(df):
    raw = df["value_raw"].astype(float)
    fill = pd.Series(False, index=df.index)
    for v in conf_fill_values:
        fill |= np.isclose(raw, v, rtol=0.0, atol=1e-6 * max(1.0, abs(v)))
    for attr in ("fillvalue", "missing_value"):
        if attr in df.columns:
            ref = pd.to_numeric(df[attr], errors="coerce")
            fill |= ref.notna() & np.isclose(raw, ref.fillna(0.0))
    return fill


validated, report, frames = [], [], []
for d in downloads:
    df = pd.read_csv(d["file"], sep=conf_delimiter_csv, dtype={"site": str})
    df["TIME"] = pd.to_datetime(df["TIME"])
    df["value"], df["units"] = canonical(df["value_raw"].astype(float), df["units_raw"])
    rng = df["standard_name"].map(lambda sn: tuple(conf_valid_ranges.get(sn, ("", -math.inf, math.inf))))
    has_range = df["standard_name"].isin(list(conf_valid_ranges))
    in_range = pd.Series(True, index=df.index)
    if has_range.any():
        unit_ok = df["units"] == rng.map(lambda r: r[0])
        low = rng.map(lambda r: r[1]).astype(float)
        high = rng.map(lambda r: r[2]).astype(float)
        in_range = ~has_range | (unit_ok & df["value"].between(low, high))
    fill = is_fill(df)
    df["quality"] = np.where(df["value"].isna(), "missing",
                             np.where(fill, "fill_value", np.where(in_range, "good", "out_of_range")))
    path = d["file"].replace(".csv", ".validated.csv")
    # one time format for every source: ISO 8601, UTC, whole seconds
    df.assign(TIME=df["TIME"].dt.round("s").dt.strftime("%Y-%m-%dT%H:%M:%SZ")).to_csv(
        path, sep=conf_delimiter_csv, index=False)
    validated.append(d | {"validated_file": path})
    good = df[df["quality"] == "good"]
    report.append({
        "RI": d["ri"], "table": d["table"], "variable": d["column"], "standard_name": d["standard_name"],
        "rows": len(df), "good": len(good),
        "fill_value": int((df["quality"] == "fill_value").sum()),
        "out_of_range": int((df["quality"] == "out_of_range").sum()),
        "missing": int((df["quality"] == "missing").sum()),
        "good_pct": round(100.0 * len(good) / len(df), 2) if len(df) else 0.0,
        "range_checked": bool(has_range.any()),
        "units_raw": ";".join(sorted(df["units_raw"].astype(str).unique())),
        "units": ";".join(sorted(good["units"].astype(str).unique())),
        "sites": good["site"].nunique(),
        "first": good["TIME"].min(), "last": good["TIME"].max(),
        "min": round(good["value"].min(), 3) if len(good) else None,
        "mean": round(good["value"].mean(), 3) if len(good) else None,
        "max": round(good["value"].max(), 3) if len(good) else None,
    })
    if len(good):
        frames.append(good.assign(source=f"{d['ri']} {d['table']}.{d['column']}"))

columns_out = ["RI", "table", "variable", "standard_name", "rows", "good", "fill_value", "out_of_range", "missing",
               "good_pct", "range_checked", "units_raw", "units", "sites", "first", "last", "min", "mean", "max"]
validation = pd.DataFrame(report, columns=columns_out)
file_validation = os.path.join(out_dir, f"{conf_vlab_name}-5-validation.{exv_code}.csv")
validation.to_csv(file_validation, index=False)
file_validated = os.path.join(out_dir, f"{conf_vlab_name}-5-validated.{exv_code}.json")
with open(file_validated, "w", encoding="utf-8", newline="\n") as fh:
    json.dump(validated, fh, indent=2)
    fh.write("\n")

good_all = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(
    columns=["TIME", "LATITUDE", "LONGITUDE", "value", "units", "standard_name", "source", "RI"])
names = sorted(good_all["standard_name"].unique())
sources = sorted(good_all["source"].unique())
markers = ["s", "^", "o", "D", "v", "P", "X", "*"]
colors = plt.get_cmap("tab10")


def no_data(ax):
    ax.text(0.5, 0.5, f"No valid {exv_code} data from a token-free\nresearch infrastructure in this window",
            ha="center", va="center", transform=ax.transAxes)
    ax.set_axis_off()


# Map: where each source has valid observations (as the Beacon_ECV17 notebook, without web tiles)
fig, ax = plt.subplots(figsize=(10, 8), dpi=100)
if sources:
    for i, source in enumerate(sources):
        g = good_all[good_all["source"] == source].drop_duplicates(["LONGITUDE", "LATITUDE"])
        ax.scatter(g["LONGITUDE"], g["LATITUDE"], s=40 if g["LONGITUDE"].nunique() < 50 else 6,
                   marker=markers[i % len(markers)], color=colors(i % 10), edgecolors="black",
                   linewidths=0.2, label=source, alpha=0.8)
    ax.grid(True, linewidth=0.3)
    ax.legend(title="Source (RI table.variable)", fontsize=7)
    ax.set_xlabel("lon")
    ax.set_ylabel("lat")
else:
    no_data(ax)
ax.set_title(f"{exv_code} {exv_label}: valid observations")
fig.tight_layout()
file_map = os.path.join(out_dir, f"{conf_vlab_name}-5-map.{exv_code}.png")
fig.savefig(file_map, metadata={"Software": None})
plt.close(fig)

# Time series and value distributions: one panel per CF name, one line per source
n = max(1, len(names))
fig_t, axes_t = plt.subplots(n, 1, figsize=(11, 3.2 * n), dpi=100, squeeze=False)
fig_h, axes_h = plt.subplots(n, 1, figsize=(9, 3.0 * n), dpi=100, squeeze=False)
if not names:
    no_data(axes_t[0][0])
    no_data(axes_h[0][0])
for k, sn in enumerate(names):
    at, ah = axes_t[k][0], axes_h[k][0]
    sub = good_all[good_all["standard_name"] == sn]
    unit = ";".join(sorted(sub["units"].astype(str).unique()))
    values = sub["value"].astype(float)
    lo, hi = np.nanpercentile(values, [0.5, 99.5]) if len(values) else (0.0, 1.0)
    bins = np.linspace(lo, hi if hi > lo else lo + 1.0, 50)
    for i, source in enumerate(sorted(sub["source"].unique())):
        g = sub[sub["source"] == source]
        daily = g.set_index("TIME")["value"].astype(float).resample("D").mean()
        at.plot(daily.index, daily.values, marker=".", linewidth=0.8, label=source)
        ah.hist(g["value"].astype(float).clip(bins[0], bins[-1]), bins=bins, density=True,
                histtype="step", linewidth=1.5, label=source)
    at.set_ylabel(f"[{unit}]")
    at.set_title(f"{sn}: daily mean per source", fontsize=9)
    at.grid(True, linewidth=0.3)
    at.legend(fontsize=7)
    ah.set_xlabel(f"{sn} [{unit}]", fontsize=8)
    ah.set_ylabel("density")
    ah.grid(True, linewidth=0.3)
    ah.legend(fontsize=7)
fig_t.suptitle(f"{exv_code} {exv_label}")
fig_h.suptitle(f"{exv_code} {exv_label}: value distribution per source")
fig_t.tight_layout()
fig_h.tight_layout()
file_timeseries = os.path.join(out_dir, f"{conf_vlab_name}-5-timeseries.{exv_code}.png")
file_histogram = os.path.join(out_dir, f"{conf_vlab_name}-5-histogram.{exv_code}.png")
fig_t.savefig(file_timeseries, metadata={"Software": None})
fig_h.savefig(file_histogram, metadata={"Software": None})
plt.close(fig_t)
plt.close(fig_h)

print(validation.to_string(index=False) if len(validation) else "No sources to validate")


## 6. Combine all valid data into one CSV


In [ ]:
# ECVs combine datasets v2
# ---
# NaaVRE:
#  cell:
#   inputs:
#    - exv_code: String
#    - file_exv: String
#    - file_discovery: String
#    - file_validated: String
#    - file_validation: String
#    - file_map: String
#    - file_timeseries: String
#    - file_histogram: String
#    - start_date: String
#    - end_date: String
#    - lon_min: Float
#    - lon_max: Float
#    - lat_min: Float
#    - lat_max: Float
#    - height_min: Float
#    - height_max: Float
#   outputs:
#    - file_combined: String
# ...
import hashlib
import json
import os

try:
    import pandas as pd
except ImportError as exc:
    raise RuntimeError("pandas must be installed") from exc

with open(file_exv, encoding="utf-8") as fh:
    nerc = json.load(fh)
with open(file_validated, encoding="utf-8") as fh:
    validated = json.load(fh)
discovery = pd.read_csv(file_discovery)
out_dir = os.path.join(conf_minio_user_local_root, exv_code)

columns = ["TIME", "LATITUDE", "LONGITUDE", "HEIGHT", "RI", "table", "site", "variable",
           "standard_name", "value", "units", "value_raw", "units_raw", "matrix"]
parts = []
for v in validated:
    df = pd.read_csv(v["validated_file"], sep=conf_delimiter_csv, dtype={"site": str})
    if "matrix" not in df.columns:
        df["matrix"] = ""
    good = df[df["quality"] == "good"][columns]
    if len(good):
        parts.append(good)
combined = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame(columns=columns)
combined.insert(0, "EXV", exv_code)
combined = combined.sort_values(by=["TIME", "standard_name", "RI", "table", "site", "LATITUDE", "LONGITUDE",
                                    "HEIGHT", "value"], kind="mergesort").reset_index(drop=True)
file_combined = os.path.join(out_dir, f"{conf_vlab_name}-6-combined.{exv_code}.csv")
combined.to_csv(file_combined, sep=conf_delimiter_csv, index=False)

monthly = (combined.assign(month=combined["TIME"].astype(str).str[:7])
           .groupby(["standard_name", "units", "RI", "table", "variable", "month"])["value"]
           .agg(["size", "mean", "min", "max"]).round(3).reset_index()) if len(combined) else pd.DataFrame(
    columns=["standard_name", "units", "RI", "table", "variable", "month", "size", "mean", "min", "max"])
file_monthly = os.path.join(out_dir, f"{conf_vlab_name}-6-monthly.{exv_code}.csv")
monthly.to_csv(file_monthly, sep=conf_delimiter_csv, index=False)

status = discovery["status"].astype(str)
token = sorted(set(discovery.loc[status.str.startswith("skipped: the node needs"), "ri"]))
unreachable = sorted(set(discovery.loc[status.str.startswith("error"), "table"]))
notes = ([f"not searched, token needed: {', '.join(token)}"] if token else []) +     ([f"errors reaching: {', '.join(unreachable)}"] if unreachable else [])
if len(combined):
    outcome = f"{len(combined)} valid rows from {len(validated)} sources"
elif not nerc["cf_standard_names"]:
    outcome = "no data: NERC gives no CF standard name for this EXV"
elif unreachable:
    outcome = "no data from the research infrastructures that could be reached"
else:
    outcome = "no data: no token-free research infrastructure serves these CF names in the window"
if notes:
    outcome += f" ({'; '.join(notes)})"
provenance = {
    "exv": nerc["exv"],
    "query": nerc["query"],
    "cf_source": nerc["cf_source"],
    "cf_standard_names_searched": nerc["cf_standard_names"],
    "window": {"lon": [lon_min, lon_max], "lat": [lat_min, lat_max],
               "time": [start_date, end_date], "height_m": [height_min, height_max]},
    "outcome": outcome,
    "sources": [{"ri": v["ri"], "table": v["table"], "variable": v["column"], "standard_name": v["standard_name"],
                 "rows_from_beacon": v["rows_from_beacon"], "rows_in_height_range": v["rows_in_height_range"],
                 "rows_combined": int(((combined["table"] == v["table"]) & (combined["variable"] == v["column"])
                                       & (combined["standard_name"] == v["standard_name"])).sum())}
                for v in validated],
    "rows_combined": len(combined),
    "columns": {
        "value": "measurement in the canonical unit (degC, hPa, ppb, ppm, m/s, ...); see units",
        "value_raw": "measurement as served by Beacon, in units_raw",
        "HEIGHT": "metres: station altitude (ACTRIS) or barometric altitude (IAGOS)",
        "site": "ACTRIS station code or IAGOS flight name",
        "matrix": "the RI's medium tag (ACTRIS ebas_matrix), empty if none",
    },
}
file_provenance = os.path.join(out_dir, f"{conf_vlab_name}-6-provenance.{exv_code}.json")
with open(file_provenance, "w", encoding="utf-8", newline="\n") as fh:
    json.dump(provenance, fh, indent=2, ensure_ascii=False, default=str)
    fh.write("\n")

files = [file_exv, file_discovery] + [v["file"] for v in validated] + \
    [file_validation, file_map, file_timeseries, file_histogram, file_combined, file_monthly, file_provenance]
lines = [f"# {conf_vlab_name} {exv_code} ({nerc['exv']['label']}) results", "", f"Outcome: {outcome}", "",
         "| File | Bytes | MD5 |", "| --- | --- | --- |"]
for path in files:
    with open(path, "rb") as fh:
        data = fh.read()
    lines.append(f"| {os.path.basename(path)} | {len(data)} | {hashlib.md5(data).hexdigest()} |")
lines.append("")
with open(os.path.join(out_dir, "log.md"), "w", encoding="utf-8", newline="\n") as fh:
    fh.write("\n".join(lines))

print(f"{exv_code}: {outcome} -> {file_combined}")
print("\n".join(lines))


## Results

Everything is in `Cloud Storage/naa-vre-user-data/ECVs/<EXV code>/`. The file
for further use is `ECVs-6-combined.<EXV>.csv`; `ECVs-6-provenance.<EXV>.json`
records the NERC terms, the window, the sources behind it and the outcome, and
`log.md` lists every file with its MD5.
